# Comprehensive EDA of Brazilian Human Rights Hotline Complaints

## Step 7: Geographic Analysis

This notebook examines the spatial distribution of recorded complaints, victims, and suspects in the cleaned Brazilian human rights hotline dataset. It validates country-UF-municipality relationships, distinguishes volume from population-standardized rates, measures geographic concentration, maps completeness separately from occurrence, and applies disclosure controls to small cells.

### Main questions

1. How complete and consistent are the geographic fields?
2. How are recorded complaints distributed across Brazilian states and municipalities?
3. How do complaint location, victim location, and suspect location differ?
4. Which areas have high reporting volume, and which have high population-standardized rates?
5. How concentrated are recorded complaints geographically?
6. Do geographic patterns change over time or by violation category?
7. Which results require suppression because of small counts?

> Geographic counts represent recorded hotline activity. They are affected by population size, access to reporting channels, digital inclusion, awareness, service availability, institutional practices, and data completeness. They must not be interpreted as population risk without appropriate denominators.


## 7.1 Imports and configuration

The notebook runs with the cleaned dataset from Step 3. Optional IBGE reference files enable municipality validation, population denominators, state codes, and choropleth maps.

### Expected optional reference files

- `ibge_municipalities.csv`: municipality code, official municipality name, UF, and state code
- `ibge_population.csv`: geography code, year, and population
- `brazil_states.geojson`: state polygons with an IBGE state-code property
- `brazil_municipalities.geojson`: municipality polygons with an IBGE municipality-code property

Use year-aligned population estimates. Do not combine a complaint period with a denominator from an unrelated year without documenting the assumption.


In [1]:
from pathlib import Path
import json
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import entropy
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 180)

PARQUET_PATH = Path("outputs/step_03_data_cleaning/disque100_clean.parquet")
CSV_PATH = Path("outputs/step_03_data_cleaning/disque100_clean.csv")
OUTPUT_DIR = Path("outputs/step_07_geographic_analysis")
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

IBGE_MUNICIPALITY_PATH = Path("reference/ibge_municipalities.csv")
IBGE_POPULATION_PATH = Path("reference/ibge_population.csv")
STATE_GEOJSON_PATH = Path("reference/brazil_states.geojson")
MUNICIPALITY_GEOJSON_PATH = Path("reference/brazil_municipalities.geojson")

COUNTING_MODE = "record"  # record or complaint
IDENTIFIER = "hash"
DATE_COLUMN = "data_de_cadastro"
RATE_MULTIPLIER = 100_000
SUPPRESSION_THRESHOLD = 5
TOP_N_MUNICIPALITIES = 30
TOP_N_VIOLATIONS = 10
EXPORT_FIGURES_HTML = True
PLOTLY_TEMPLATE = "plotly_white"

UF_CODES = {"AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"}


## 7.2 Load the cleaned dataset and define the analytical unit


In [2]:
def load_clean_dataset():
    if PARQUET_PATH.exists():
        return pd.read_parquet(PARQUET_PATH), PARQUET_PATH
    if CSV_PATH.exists():
        return pd.read_csv(CSV_PATH, low_memory=False), CSV_PATH
    raise FileNotFoundError("Run Step 3 or update PARQUET_PATH/CSV_PATH.")


df, DATA_SOURCE = load_clean_dataset()
if DATE_COLUMN in df.columns:
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
if COUNTING_MODE not in {"record", "complaint"}:
    raise ValueError("COUNTING_MODE must be 'record' or 'complaint'.")
if COUNTING_MODE == "complaint":
    if IDENTIFIER not in df.columns:
        raise ValueError("Complaint mode requires hash.")
    df_analysis = df.dropna(subset=[IDENTIFIER]).sort_values(DATE_COLUMN).drop_duplicates(IDENTIFIER).copy()
    unit_label = "complaints"
else:
    df_analysis = df.copy()
    unit_label = "records"

print("Source:", DATA_SOURCE.resolve())
print(f"Analytical population: {len(df_analysis):,} {unit_label}")


Source: C:\Projetos\eda-disque100-completo\eda\outputs\step_03_data_cleaning\disque100_clean.parquet
Analytical population: 8,401 records


## 7.3 Geographic roles and fields

Three spatial concepts are retained separately:

- **Complaint location:** where the reported violation occurred or was registered, depending on source semantics
- **Victim location:** victim country, UF, and municipality
- **Suspect location:** suspect country, UF, and municipality

Confirm these semantics against the official data dictionary before drawing spatial conclusions.


In [3]:
GEOGRAPHIC_ROLES = {
    "complaint": {"country":"pais", "state":"uf", "municipality":"municipio"},
    "victim": {"country":"pais_da_vitima", "state":"uf_da_vitima", "municipality":"municipio_da_vitima"},
    "suspect": {"country":"pais_do_suspeito", "state":"uf_do_suspeito", "municipality":"sl_suspeito_municipio"},
}

role_rows=[]
for role, fields in GEOGRAPHIC_ROLES.items():
    for level, column in fields.items():
        role_rows.append({"role":role,"level":level,"column":column,"present":column in df_analysis.columns})
geographic_field_inventory=pd.DataFrame(role_rows)
display(geographic_field_inventory)


,role,level,column,present
0,complaint,country,pais,True
1,complaint,state,uf,True
2,complaint,municipality,municipio,True
3,victim,country,pais_da_vitima,True
4,victim,state,uf_da_vitima,True
5,victim,municipality,municipio_da_vitima,True
6,suspect,country,pais_do_suspeito,True
7,suspect,state,uf_do_suspeito,True
8,suspect,municipality,sl_suspeito_municipio,True


## 7.4 Canonical geographic matching

Canonical keys normalize Unicode, spacing, case, and accents for matching only. Original Portuguese labels remain available for display.


In [4]:
def strip_accents(text):
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def geo_key(value):
    if pd.isna(value):
        return pd.NA
    text=unicodedata.normalize("NFKC",str(value))
    text=re.sub(r"\s+"," ",text.strip()).upper()
    return strip_accents(text)


def normalize_uf(value):
    key=geo_key(value)
    return key if pd.notna(key) else pd.NA


def normalize_municipality(value):
    key=geo_key(value)
    if pd.isna(key): return pd.NA
    key=re.sub(r"\s+-\s+[A-Z]{2}$","",key)
    key=re.sub(r"\s+/\s+[A-Z]{2}$","",key)
    return key


## 7.5 Geographic completeness

Completeness is calculated independently for complaint, victim, and suspect geography. Missing percentages should be reported with the applicable denominator and not confused with absence of the geographic entity.


In [5]:
completeness_rows=[]
for role, fields in GEOGRAPHIC_ROLES.items():
    for level, column in fields.items():
        if column in df_analysis.columns:
            missing=int(df_analysis[column].isna().sum())
            completeness_rows.append({
                "role":role,"level":level,"column":column,"total":len(df_analysis),
                "valid":int(df_analysis[column].notna().sum()),"missing":missing,
                "missing_pct":round(missing/len(df_analysis)*100,2),
                "unique_valid":int(df_analysis[column].nunique(dropna=True))
            })
geographic_completeness=pd.DataFrame(completeness_rows)
display(geographic_completeness)

fig=px.bar(geographic_completeness,x="missing_pct",y="column",orientation="h",color="role",text="missing_pct",title="Missingness of geographic fields",labels={"missing_pct":"Missing (%)"})
fig.update_layout(template=PLOTLY_TEMPLATE)
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"geographic_completeness.html",include_plotlyjs=True)


,role,level,column,total,valid,missing,missing_pct,unique_valid
0,complaint,country,pais,8401,8401,0,0.00,29
1,complaint,state,uf,8401,7828,573,6.82,30
2,complaint,municipality,municipio,8401,7810,591,7.03,313
3,victim,country,pais_da_vitima,8401,8401,0,0.00,22
4,victim,state,uf_da_vitima,8401,7813,588,7.00,30
5,victim,municipality,municipio_da_vitima,8401,7760,641,7.63,301
6,suspect,country,pais_do_suspeito,8401,8401,0,0.00,30
7,suspect,state,uf_do_suspeito,8401,7942,459,5.46,30
8,suspect,municipality,sl_suspeito_municipio,8401,7896,505,6.01,327


## 7.6 Country-UF-municipality logical consistency

Checks include:

- Brazil with missing UF
- non-Brazil country with Brazilian UF populated
- invalid UF codes
- valid UF with missing municipality

Findings are review items because field applicability may differ by form version.


In [6]:
geo_logic_rows=[]
BRAZIL_KEYS={"BRASIL","BRAZIL","BR"}
for role, fields in GEOGRAPHIC_ROLES.items():
    country_col,state_col,municipality_col=fields["country"],fields["state"],fields["municipality"]
    if country_col not in df_analysis.columns: continue
    country=df_analysis[country_col].map(geo_key)
    brazil=country.isin(BRAZIL_KEYS)
    if state_col in df_analysis.columns:
        state=df_analysis[state_col].map(normalize_uf)
        state_missing=state.isna()
        valid_state=state.isin(UF_CODES)
        geo_logic_rows.extend([
            {"role":role,"rule":"Brazil requires UF review","violations":int((brazil & state_missing).sum())},
            {"role":role,"rule":"UF must be one of 27 official codes","violations":int((~state_missing & ~valid_state).sum())},
            {"role":role,"rule":"Non-Brazil country with Brazilian UF","violations":int((~brazil & country.notna() & valid_state).sum())},
        ])
        if municipality_col in df_analysis.columns:
            municipality_missing=df_analysis[municipality_col].isna()
            geo_logic_rows.append({"role":role,"rule":"Valid UF with missing municipality","violations":int((valid_state & municipality_missing).sum())})
geographic_logic_checks=pd.DataFrame(geo_logic_rows)
geographic_logic_checks["percent_rows"]=(geographic_logic_checks["violations"]/len(df_analysis)*100).round(2)
display(geographic_logic_checks)


,role,rule,violations,percent_rows
0,complaint,Brazil requires UF review,0,0.00
1,complaint,UF must be one of 27 official codes,927,11.03
2,complaint,Non-Brazil country with Brazilian UF,6096,72.56
3,complaint,Valid UF with missing municipality,0,0.00
4,victim,Brazil requires UF review,0,0.00
5,victim,UF must be one of 27 official codes,1416,16.86
6,victim,Non-Brazil country with Brazilian UF,5598,66.63
7,victim,Valid UF with missing municipality,53,0.63
8,suspect,Brazil requires UF review,0,0.00
9,suspect,UF must be one of 27 official codes,1202,14.31


## 7.7 Load and validate optional IBGE reference tables

The code expects standardized columns but includes aliases. Review the detected column mapping before relying on municipality validation or rates.


In [7]:
def first_existing(columns, candidates):
    for candidate in candidates:
        if candidate in columns: return candidate
    return None

ibge_municipalities=pd.DataFrame()
municipality_reference_status="not_available"
if IBGE_MUNICIPALITY_PATH.exists():
    ibge_municipalities=pd.read_csv(IBGE_MUNICIPALITY_PATH,low_memory=False)
    ibge_municipalities.columns=[str(c).strip().lower() for c in ibge_municipalities.columns]
    code_col=first_existing(ibge_municipalities.columns,["municipality_code","codigo_municipio","cod_municipio","id_municipio","codigo_ibge"])
    name_col=first_existing(ibge_municipalities.columns,["municipality_name","municipio","nome_municipio","nome"])
    uf_col=first_existing(ibge_municipalities.columns,["uf","sigla_uf"])
    state_code_col=first_existing(ibge_municipalities.columns,["state_code","codigo_uf","cod_uf"])
    required=[code_col,name_col,uf_col]
    if any(x is None for x in required):
        raise ValueError("IBGE municipality file must contain municipality code, municipality name, and UF columns.")
    rename={code_col:"ibge_municipality_code",name_col:"ibge_municipality_name",uf_col:"ibge_uf"}
    if state_code_col: rename[state_code_col]="ibge_state_code"
    ibge_municipalities=ibge_municipalities.rename(columns=rename)
    ibge_municipalities["municipality_key"]=ibge_municipalities["ibge_municipality_name"].map(normalize_municipality)
    ibge_municipalities["uf_key"]=ibge_municipalities["ibge_uf"].map(normalize_uf)
    ibge_municipalities["ibge_municipality_code"]=ibge_municipalities["ibge_municipality_code"].astype("string").str.replace(r"\.0$","",regex=True).str.zfill(7)
    municipality_reference_status="loaded"
    display(ibge_municipalities.head())
else:
    display(Markdown("**IBGE municipality reference not loaded.** Name-UF validation and municipality maps will be skipped."))


**IBGE municipality reference not loaded.** Name-UF validation and municipality maps will be skipped.

## 7.8 Validate municipality-UF combinations against IBGE

Matching uses both normalized municipality name and UF, preventing municipalities with identical names in different states from being conflated. Unmatched values are exported for manual mapping rather than guessed.


In [8]:
municipality_validation_tables=[]
matched_geographies={}
if not ibge_municipalities.empty:
    reference=ibge_municipalities.drop_duplicates(["municipality_key","uf_key"])
    for role,fields in GEOGRAPHIC_ROLES.items():
        state_col,muni_col=fields["state"],fields["municipality"]
        if state_col not in df_analysis.columns or muni_col not in df_analysis.columns: continue
        temp=df_analysis[[state_col,muni_col]].copy()
        temp["uf_key"]=temp[state_col].map(normalize_uf)
        temp["municipality_key"]=temp[muni_col].map(normalize_municipality)
        temp=temp.merge(reference,on=["municipality_key","uf_key"],how="left",indicator=True)
        temp["role"]=role
        municipality_validation_tables.append(temp)
        matched_geographies[role]=temp
municipality_validation=pd.concat(municipality_validation_tables,ignore_index=True) if municipality_validation_tables else pd.DataFrame()
if not municipality_validation.empty:
    municipality_match_summary=municipality_validation.groupby(["role","_merge"],observed=False).size().rename("records").reset_index()
    municipality_match_summary["percent_within_role"]=(municipality_match_summary["records"]/municipality_match_summary.groupby("role")["records"].transform("sum")*100).round(2)
    unmatched_municipalities=municipality_validation[municipality_validation["_merge"]=="left_only"].groupby(["role","uf_key","municipality_key"]).size().rename("records").reset_index().sort_values("records",ascending=False)
else:
    municipality_match_summary=pd.DataFrame()
    unmatched_municipalities=pd.DataFrame()
display(municipality_match_summary)
display(unmatched_municipalities.head(100))


""


""


## 7.9 State-level volume

State counts are shown for complaint, victim, and suspect locations. Missing and invalid UF values are excluded from ranked volumes and retained in the completeness reports.


In [9]:
state_volume_tables=[]
for role,fields in GEOGRAPHIC_ROLES.items():
    state_col=fields["state"]
    if state_col not in df_analysis.columns: continue
    temp=df_analysis[state_col].map(normalize_uf)
    valid_state=temp[temp.isin(UF_CODES)]
    counts=valid_state.value_counts().rename_axis("uf").reset_index(name="count")
    counts["percent_valid_geography"]=(counts["count"]/counts["count"].sum()*100).round(2)
    counts["role"]=role
    state_volume_tables.append(counts)
state_volumes=pd.concat(state_volume_tables,ignore_index=True) if state_volume_tables else pd.DataFrame()
display(state_volumes)

for role in state_volumes["role"].unique() if not state_volumes.empty else []:
    plot=state_volumes[state_volumes["role"]==role].sort_values("count")
    fig=px.bar(plot,x="count",y="uf",orientation="h",text="count",title=f"{role.title()} geography: recorded volume by UF",labels={"count":unit_label.title()})
    fig.update_layout(template=PLOTLY_TEMPLATE)
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/f"state_volume_{role}.html",include_plotlyjs=True)


,uf,count,percent_valid_geography,role
0,SP,1797,26.04,complaint
1,MG,975,14.13,complaint
2,RJ,611,8.85,complaint
3,RS,514,7.45,complaint
4,CE,325,4.71,complaint
5,BA,287,4.16,complaint
6,PR,276,4.0,complaint
7,PE,231,3.35,complaint
8,AM,210,3.04,complaint
9,GO,200,2.9,complaint


## 7.10 Municipality-level volume and disclosure control

Municipality outputs suppress counts below 5 by default. Suppression is applied to published tables, while unsuppressed data should remain access-controlled.


In [10]:
def suppress_small_counts(table,count_col="count",threshold=SUPPRESSION_THRESHOLD):
    result=table.copy()
    result["suppressed"]=result[count_col].between(1,threshold-1)
    result["display_count"]=result[count_col].where(~result["suppressed"],pd.NA)
    result["display_label"]=np.where(result["suppressed"],f"<{threshold}",result[count_col].astype(str))
    return result

municipality_volume_tables=[]
for role,fields in GEOGRAPHIC_ROLES.items():
    state_col,muni_col=fields["state"],fields["municipality"]
    if state_col not in df_analysis.columns or muni_col not in df_analysis.columns: continue
    temp=df_analysis[[state_col,muni_col]].dropna().copy()
    temp["uf"]=temp[state_col].map(normalize_uf)
    temp["municipality_key"]=temp[muni_col].map(normalize_municipality)
    temp=temp[temp["uf"].isin(UF_CODES)]
    counts=temp.groupby(["uf","municipality_key"]).size().rename("count").reset_index()
    counts["role"]=role
    municipality_volume_tables.append(counts)
municipality_volumes=pd.concat(municipality_volume_tables,ignore_index=True) if municipality_volume_tables else pd.DataFrame()
municipality_volumes_suppressed=suppress_small_counts(municipality_volumes) if not municipality_volumes.empty else pd.DataFrame()
display(municipality_volumes_suppressed.sort_values("count",ascending=False).head(100))

for role in municipality_volumes["role"].unique() if not municipality_volumes.empty else []:
    top=municipality_volumes[(municipality_volumes["role"]==role)&(municipality_volumes["count"]>=SUPPRESSION_THRESHOLD)].nlargest(TOP_N_MUNICIPALITIES,"count").sort_values("count")
    top["place"]=top["municipality_key"]+" / "+top["uf"]
    fig=px.bar(top,x="count",y="place",orientation="h",text="count",title=f"Top municipalities: {role} geography",labels={"count":unit_label.title()})
    fig.update_layout(template=PLOTLY_TEMPLATE,height=max(500,24*len(top)+160))
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/f"top_municipalities_{role}.html",include_plotlyjs=True)


,uf,municipality_key,count,role,suppressed,display_count,display_label
406,MG,3170206 | UBERLANDIA,312,victim,False,312.0,312
91,MG,3170206 | UBERLANDIA,306,complaint,False,306.0,306
709,MG,3170206 | UBERLANDIA,306,suspect,False,306.0,306
263,SP,351550 | FERNANDOPOLIS,280,complaint,False,280.0,280
565,SP,351550 | FERNANDOPOLIS,280,victim,False,280.0,280
892,SP,351550 | FERNANDOPOLIS,271,suspect,False,271.0,271
924,SP,3550308 | SAO PAULO,255,suspect,False,255.0,255
80,MG,3136702 | JUIZ DE FORA,249,complaint,False,249.0,249
392,MG,3136702 | JUIZ DE FORA,249,victim,False,249.0,249
697,MG,3136702 | JUIZ DE FORA,246,suspect,False,246.0,246


## 7.11 Load population denominators

Population rates are produced only when an external denominator file is available. Required concepts are geography code, year, and population. The notebook does not fabricate or interpolate missing denominators.


In [22]:
population=pd.DataFrame()
population_status="not_available"
if IBGE_POPULATION_PATH.exists():
    population=pd.read_csv(IBGE_POPULATION_PATH,low_memory=False)
    population.columns=[str(c).strip().lower() for c in population.columns]
    geo_code_col=first_existing(population.columns,["geography_code","codigo_geografia","codigo_ibge","municipality_code","codigo_municipio","state_code","codigo_uf"])
    year_col=first_existing(population.columns,["year","ano"])
    pop_col=first_existing(population.columns,["population","populacao"])
    level_col=first_existing(population.columns,["geography_level","nivel_geografico","level"])
    if any(x is None for x in [geo_code_col,year_col,pop_col]):
        raise ValueError("Population file must contain geography code, year, and population.")
    rename={geo_code_col:"geography_code",year_col:"year",pop_col:"population"}
    if level_col: rename[level_col]="geography_level"
    population=population.rename(columns=rename)
    population["geography_code"]=population["geography_code"].astype("string").str.replace(r"\.0$","",regex=True)
    population["year"]=pd.to_numeric(population["year"],errors="coerce").astype("Int64")
    population["population"]=pd.to_numeric(population["population"],errors="coerce")
    population_status="loaded"
    display(population.head())
else:
    display(Markdown("**Population denominator not loaded.** Rate calculations will be skipped; only recorded volumes and shares will be reported."))


**Population denominator not loaded.** Rate calculations will be skipped; only recorded volumes and shares will be reported.

## 7.12 Population-standardized state rates

State rates require an IBGE state code. If the municipality reference includes state codes, the notebook creates a UF-to-code bridge. Rates are reported per 100,000 residents and only for matching years.


In [12]:
state_rates=pd.DataFrame()
rate_unmatched=pd.DataFrame()
if not population.empty and DATE_COLUMN in df_analysis.columns and not ibge_municipalities.empty and "ibge_state_code" in ibge_municipalities.columns:
    uf_bridge=ibge_municipalities[["uf_key","ibge_state_code"]].dropna().drop_duplicates()
    uf_bridge["geography_code"]=uf_bridge["ibge_state_code"].astype("string").str.replace(r"\.0$","",regex=True).str.zfill(2)
    temp=df_analysis.dropna(subset=[DATE_COLUMN]).copy()
    temp["year"]=temp[DATE_COLUMN].dt.year.astype("Int64")
    temp["uf_key"]=temp["uf"].map(normalize_uf) if "uf" in temp.columns else pd.NA
    counts=temp[temp["uf_key"].isin(UF_CODES)].groupby(["year","uf_key"]).size().rename("count").reset_index().merge(uf_bridge,on="uf_key",how="left")
    pop_state=population.copy()
    if "geography_level" in pop_state.columns:
        pop_state=pop_state[pop_state["geography_level"].astype(str).str.lower().isin(["state","uf","estado"])]
    pop_state["geography_code"]=pop_state["geography_code"].str.zfill(2)
    state_rates=counts.merge(pop_state[["geography_code","year","population"]],on=["geography_code","year"],how="left",indicator=True)
    state_rates["rate_per_100k"]=state_rates["count"]/state_rates["population"]*RATE_MULTIPLIER
    rate_unmatched=state_rates[state_rates["_merge"]!="both"].copy()
    state_rates=suppress_small_counts(state_rates)
    display(state_rates)
else:
    display(Markdown("**State rates skipped.** Load compatible population denominators and an IBGE UF-code bridge."))


**State rates skipped.** Load compatible population denominators and an IBGE UF-code bridge.

## 7.13 State choropleth maps

Maps require a GeoJSON whose feature property matches the configured geography code. Inspect the GeoJSON before changing `featureidkey`. Counts and rates are mapped separately because they answer different questions.


In [13]:
state_geojson=None
STATE_FEATURE_ID_KEY="properties.code"  # adapt to the GeoJSON, e.g. properties.CD_UF
if STATE_GEOJSON_PATH.exists():
    state_geojson=json.loads(STATE_GEOJSON_PATH.read_text(encoding="utf-8"))

if state_geojson is not None and not state_volumes.empty and not ibge_municipalities.empty and "ibge_state_code" in ibge_municipalities.columns:
    bridge=ibge_municipalities[["uf_key","ibge_state_code"]].drop_duplicates()
    bridge["geo_code"]=bridge["ibge_state_code"].astype("string").str.replace(r"\.0$","",regex=True).str.zfill(2)
    complaint_states=state_volumes[state_volumes["role"]=="complaint"].merge(bridge,left_on="uf",right_on="uf_key",how="left")
    fig=px.choropleth(complaint_states,geojson=state_geojson,locations="geo_code",featureidkey=STATE_FEATURE_ID_KEY,color="count",hover_name="uf",color_continuous_scale="Blues",title="Recorded complaint volume by UF")
    fig.update_geos(fitbounds="locations",visible=False)
    fig.update_layout(template=PLOTLY_TEMPLATE)
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"state_complaint_volume_map.html",include_plotlyjs=True)

if state_geojson is not None and not state_rates.empty:
    latest_year=state_rates["year"].max()
    map_data=state_rates[(state_rates["year"]==latest_year)&(~state_rates["suppressed"])&state_rates["population"].notna()].copy()
    fig=px.choropleth(map_data,geojson=state_geojson,locations="geography_code",featureidkey=STATE_FEATURE_ID_KEY,color="rate_per_100k",hover_name="uf_key",hover_data=["count","population"],color_continuous_scale="YlOrRd",title=f"Recorded complaint rate per 100,000 residents, {latest_year}")
    fig.update_geos(fitbounds="locations",visible=False)
    fig.update_layout(template=PLOTLY_TEMPLATE)
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"state_complaint_rate_map_latest_year.html",include_plotlyjs=True)


## 7.14 Geographic concentration

The notebook calculates:

- top-1, top-5, and top-10 shares
- Herfindahl-Hirschman Index (HHI)
- normalized entropy
- effective number of equally sized geographic units (`1 / HHI`)

Concentration in counts is strongly influenced by population distribution. Compare rate concentration only when denominators are available.


In [14]:
concentration_rows=[]
for role in state_volumes["role"].unique() if not state_volumes.empty else []:
    counts=state_volumes[state_volumes["role"]==role]["count"].sort_values(ascending=False)
    shares=counts/counts.sum()
    hhi=float((shares**2).sum())
    ent=float(entropy(shares))
    normalized_entropy=ent/np.log(len(shares)) if len(shares)>1 else 0
    concentration_rows.append({
        "role":role,"states":len(shares),"top1_share_pct":round(shares.iloc[:1].sum()*100,2),
        "top5_share_pct":round(shares.iloc[:5].sum()*100,2),"top10_share_pct":round(shares.iloc[:10].sum()*100,2),
        "hhi":round(hhi,4),"normalized_entropy":round(normalized_entropy,4),
        "effective_number_of_states":round(1/hhi,2) if hhi else np.nan
    })
geographic_concentration=pd.DataFrame(concentration_rows)
display(geographic_concentration)


,role,states,top1_share_pct,top5_share_pct,top10_share_pct,hhi,normalized_entropy,effective_number_of_states
0,complaint,27,26.04,61.18,78.63,0.1137,0.7990,8.79
1,victim,27,25.20,62.48,78.26,0.1129,0.8029,8.85
2,suspect,27,26.20,61.47,79.36,0.1149,0.7979,8.70


## 7.15 Compare complaint, victim, and suspect geography

This comparison uses state shares, not raw counts, because geographic completeness differs across roles. Large differences may reflect genuine location differences or differential missingness.


In [15]:
role_state_comparison=pd.DataFrame()
if not state_volumes.empty:
    role_state_comparison=state_volumes.pivot_table(index="uf",columns="role",values="percent_valid_geography",fill_value=0).reset_index()
    display(role_state_comparison)
    long=role_state_comparison.melt(id_vars="uf",var_name="role",value_name="percent_valid_geography")
    fig=px.bar(long,x="uf",y="percent_valid_geography",color="role",barmode="group",title="State share by geographic role",labels={"percent_valid_geography":"Share of valid geography (%)"})
    fig.update_layout(template=PLOTLY_TEMPLATE)
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"state_share_by_geographic_role.html",include_plotlyjs=True)


role,uf,complaint,suspect,victim
0,AC,1.35,0.18,1.53
1,AL,0.14,0.15,0.16
2,AM,3.04,2.95,2.63
3,AP,0.2,0.21,0.22
4,BA,4.16,4.41,5.05
5,CE,4.71,4.58,2.19
6,DF,2.56,1.51,2.78
7,ES,2.0,1.62,1.56
8,GO,2.9,3.71,2.69
9,MA,0.81,1.16,0.88


## 7.16 Violation composition by UF

The heatmap uses row percentages, showing the composition of top recorded violation categories within each UF. States with small complete-case counts should not be compared without checking stability and suppression thresholds.


In [16]:
state_violation_counts=pd.DataFrame()
state_violation_row_pct=pd.DataFrame()
if {"uf","violacao"}.issubset(df_analysis.columns):
    temp=df_analysis[["uf","violacao"]].dropna().copy()
    temp["uf_key"]=temp["uf"].map(normalize_uf)
    temp=temp[temp["uf_key"].isin(UF_CODES)]
    top_violations=temp["violacao"].value_counts().head(TOP_N_VIOLATIONS).index
    temp["violation_group"]=temp["violacao"].where(temp["violacao"].isin(top_violations),"OTHER")
    state_violation_counts=pd.crosstab(temp["uf_key"],temp["violation_group"])
    eligible=state_violation_counts.sum(axis=1)>=SUPPRESSION_THRESHOLD
    state_violation_row_pct=state_violation_counts.loc[eligible].div(state_violation_counts.loc[eligible].sum(axis=1),axis=0)*100
    fig=px.imshow(state_violation_row_pct.round(1),text_auto=".1f",aspect="auto",color_continuous_scale="Blues",title="Violation composition within UF (%)",labels={"color":"Row %"})
    fig.update_layout(template=PLOTLY_TEMPLATE,height=800)
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"state_violation_composition.html",include_plotlyjs=True)


## 7.17 Geographic patterns over time

Monthly state shares reveal whether national trends are driven by a few states or whether geographic composition changes. The chart is limited to the most frequent UFs for readability.


In [17]:
state_time=pd.DataFrame()
if {"uf",DATE_COLUMN}.issubset(df_analysis.columns):
    temp=df_analysis.dropna(subset=["uf",DATE_COLUMN]).copy()
    temp["uf_key"]=temp["uf"].map(normalize_uf)
    temp=temp[temp["uf_key"].isin(UF_CODES)]
    temp["month_start"]=temp[DATE_COLUMN].dt.to_period("M").dt.to_timestamp()
    top_states=temp["uf_key"].value_counts().head(10).index
    temp["uf_group"]=temp["uf_key"].where(temp["uf_key"].isin(top_states),"OTHER")
    state_time=temp.groupby(["month_start","uf_group"]).size().rename("count").reset_index()
    state_time["monthly_total"]=state_time.groupby("month_start")["count"].transform("sum")
    state_time["share_pct"]=state_time["count"]/state_time["monthly_total"]*100
    fig=px.area(state_time,x="month_start",y="share_pct",color="uf_group",title="Monthly geographic composition of recorded complaints",labels={"share_pct":"Monthly share (%)"})
    fig.update_layout(template=PLOTLY_TEMPLATE,yaxis_range=[0,100])
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"state_composition_over_time.html",include_plotlyjs=True)


## 7.18 Geographic completeness map/table

A state with few records may have low reported volume, poor geographic completeness, or both. Mapping completeness separately prevents missing location data from being misread as low occurrence.


In [18]:
state_completeness=pd.DataFrame()
if "uf" in df_analysis.columns:
    temp=df_analysis.copy()
    temp["uf_valid"]=temp["uf"].map(normalize_uf).isin(UF_CODES)
    if DATE_COLUMN in temp.columns:
        temp["year"]=temp[DATE_COLUMN].dt.year.astype("Int64")
        state_completeness=temp.groupby("year",dropna=False)["uf_valid"].agg(total="size",valid="sum",completeness="mean").reset_index()
        state_completeness["completeness_pct"]=(state_completeness["completeness"]*100).round(2)
    else:
        state_completeness=pd.DataFrame({"total":[len(temp)],"valid":[temp["uf_valid"].sum()],"completeness_pct":[temp["uf_valid"].mean()*100]})
display(state_completeness)


,year,total,valid,completeness,completeness_pct
0,2021,568,490,0.862676,86.27
1,2022,730,626,0.857534,85.75
2,2023,1657,1497,0.903440,90.34
3,2024,1874,1660,0.885806,88.58
4,2025,3572,2628,0.735722,73.57


## 7.19 Automated geographic findings register

The register highlights missingness, invalid state codes, unmatched municipalities, suppression burden, rate-denominator gaps, concentration, and high-volume areas. Findings require contextual review.


In [23]:
finding_rows=[]
for _,r in geographic_completeness.iterrows():
    if r["missing_pct"]>=40:
        finding_rows.append({"finding_type":"high_geographic_missingness","scope":f"{r['role']}:{r['column']}","finding":f"{r['missing_pct']:.2f}% missing","interpretation":"Restrict spatial inference and report the denominator of valid geography."})
for _,r in geographic_logic_checks[geographic_logic_checks["violations"]>0].iterrows():
    finding_rows.append({"finding_type":"geographic_rule_violation","scope":r["role"],"finding":f"{r['rule']}: {int(r['violations'])} records","interpretation":"Resolve with official metadata or quarantine before detailed mapping."})
if not unmatched_municipalities.empty:
    for role,g in unmatched_municipalities.groupby("role"):
        finding_rows.append({"finding_type":"unmatched_municipality","scope":role,"finding":f"{len(g)} unique municipality-UF combinations unmatched","interpretation":"Review spelling and source semantics; do not use fuzzy matches without manual approval."})
if not municipality_volumes_suppressed.empty:
    suppressed=municipality_volumes_suppressed["suppressed"].sum()
    finding_rows.append({"finding_type":"small_cell_suppression","scope":"municipality","finding":f"{int(suppressed)} municipality-role cells below {SUPPRESSION_THRESHOLD}","interpretation":"Suppress in public outputs and consider complementary suppression."})
if population.empty:
    finding_rows.append({"finding_type":"missing_denominator","scope":"rates","finding":"No compatible population denominator loaded","interpretation":"Do not interpret raw counts as geographic risk."})
for _,r in geographic_concentration.iterrows():
    finding_rows.append({"finding_type":"geographic_concentration","scope":r["role"],"finding":f"Top five UFs contain {r['top5_share_pct']:.2f}% of valid records","interpretation":"Compare with population distribution and reporting access before interpreting concentration."})
geographic_findings=pd.DataFrame(finding_rows)
display(geographic_findings)


,finding_type,scope,finding,interpretation
0,geographic_rule_violation,complaint,UF must be one of 27 official codes: 927 records,Resolve with official metadata or quarantine before detailed mapping.
1,geographic_rule_violation,complaint,Non-Brazil country with Brazilian UF: 6096 records,Resolve with official metadata or quarantine before detailed mapping.
2,geographic_rule_violation,victim,UF must be one of 27 official codes: 1416 records,Resolve with official metadata or quarantine before detailed mapping.
3,geographic_rule_violation,victim,Non-Brazil country with Brazilian UF: 5598 records,Resolve with official metadata or quarantine before detailed mapping.
4,geographic_rule_violation,victim,Valid UF with missing municipality: 53 records,Resolve with official metadata or quarantine before detailed mapping.
5,geographic_rule_violation,suspect,UF must be one of 27 official codes: 1202 records,Resolve with official metadata or quarantine before detailed mapping.
6,geographic_rule_violation,suspect,Non-Brazil country with Brazilian UF: 5935 records,Resolve with official metadata or quarantine before detailed mapping.
7,geographic_rule_violation,suspect,Valid UF with missing municipality: 46 records,Resolve with official metadata or quarantine before detailed mapping.
8,small_cell_suppression,municipality,279 municipality-role cells below 5,Suppress in public outputs and consider complementary suppression.
9,missing_denominator,rates,No compatible population denominator loaded,Do not interpret raw counts as geographic risk.


## 7.20 Export tables, figures, and metadata


In [20]:
exports={
    "geographic_field_inventory.csv":geographic_field_inventory,
    "geographic_completeness.csv":geographic_completeness,
    "geographic_logic_checks.csv":geographic_logic_checks,
    "municipality_match_summary.csv":municipality_match_summary,
    "unmatched_municipalities.csv":unmatched_municipalities,
    "state_volumes.csv":state_volumes,
    "municipality_volumes_suppressed.csv":municipality_volumes_suppressed,
    "state_rates.csv":state_rates,
    "rate_unmatched.csv":rate_unmatched,
    "geographic_concentration.csv":geographic_concentration,
    "role_state_comparison.csv":role_state_comparison,
    "state_violation_counts.csv":state_violation_counts,
    "state_violation_row_percent.csv":state_violation_row_pct,
    "state_time.csv":state_time,
    "state_completeness.csv":state_completeness,
    "geographic_findings.csv":geographic_findings,
}
for filename,table in exports.items():
    table.to_csv(TABLE_DIR/filename,index=False,encoding="utf-8-sig")

figure_index=pd.DataFrame({"figure_file":sorted(p.name for p in FIGURE_DIR.glob("*.html"))})
figure_index.to_csv(OUTPUT_DIR/"figure_index.csv",index=False,encoding="utf-8-sig")
metadata={
    "data_source":str(DATA_SOURCE),"counting_mode":COUNTING_MODE,
    "analytical_unit":unit_label,"analytical_rows":int(len(df_analysis)),
    "municipality_reference_status":municipality_reference_status,
    "population_status":population_status,
    "rate_multiplier":RATE_MULTIPLIER,
    "suppression_threshold":SUPPRESSION_THRESHOLD,
    "state_geojson_loaded":STATE_GEOJSON_PATH.exists(),
    "municipality_geojson_loaded":MUNICIPALITY_GEOJSON_PATH.exists(),
}
(OUTPUT_DIR/"run_metadata.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")
print("Tables:",TABLE_DIR.resolve())
print("Interactive figures:",FIGURE_DIR.resolve())


Tables: C:\Projetos\eda-disque100-completo\eda\outputs\step_07_geographic_analysis\tables
Interactive figures: C:\Projetos\eda-disque100-completo\eda\outputs\step_07_geographic_analysis\figures


## 7.21 Interpretation checklist

Before reporting geographic findings:

1. Define what complaint, victim, and suspect geography mean in the source system.
2. State whether counts represent records or unique complaints.
3. Report geographic completeness for every mapped variable.
4. Validate municipality-UF pairs against official IBGE codes.
5. Keep raw volume and population-standardized rates in separate figures.
6. Align numerator and population denominator years.
7. Do not interpret high counts as high risk without denominators.
8. Consider reporting-channel access, digital inclusion, awareness, and institutional capacity.
9. Suppress cells below 5 and evaluate complementary suppression.
10. Avoid fuzzy municipality corrections unless mappings are manually reviewed.
11. Compare geographic changes with temporal changes in forms, channels, and completeness.
12. Document the GeoJSON source, version, feature keys, and coordinate reference system.

### Recommended reporting language

“Recorded complaint volume varied across Brazilian states and municipalities. Because reporting volume is influenced by population size, hotline access, awareness, service availability, and geographic completeness, raw counts were not interpreted as population risk. Where year-aligned IBGE denominators were available, rates per 100,000 residents were reported separately. Small cells were suppressed to reduce disclosure risk.”

## Next step

**Step 8: Profile and Multivariate Exploration** should integrate victim, suspect, violation, relationship, temporal, and geographic features; examine multidimensional patterns using carefully selected faceted charts and multiple correspondence analysis or clustering only when assumptions and analytical objectives justify them.
